In [ ]:
""" We will use feature extractiom approach for quick bert' model selection.
The approach features the encoder' [CLS] and a OneVsRest Logistic Regressor."""

" We will use feature extractiom approach for quick bert' model selection"

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
import psutil

/Users/mouldi/Projects/ToxicCommentsClassification/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting up the environment variables
DATA_PATH = "Finetuning/data/processed/data.csv"
TEXT_COL = "text"
LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

MODELS_TO_EVAL = [
    "roberta-base",
    "vinai/bertweet-base",
    "GroNLP/hateBERT",
    "distilbert-base-uncased",
    "bert-large-uncased"
]

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(DEVICE)

mps


In [3]:
# Importing and splitting the dataset
df = pd.read_csv(DATA_PATH)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.8, random_state=42)

for keep_idx, _ in msss.split(df, df[LABEL_COLS]):
    df_sample = df.iloc[keep_idx] # Using 1/5 of the initial dataset for quick inference

print(f"Subset size: {len(df_sample)}")
print(df_sample[LABEL_COLS].sum())

texts = df_sample['comment_text'].to_list()
Y = df_sample[LABEL_COLS].values.astype(int)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, Y, test_size=0.2, random_state=42
)

Subset size: 44709
toxic            4277
severe_toxic      392
obscene          2428
threat            138
insult           2261
identity_hate     423
dtype: int64


In [4]:
# Function to get the [CLS] embeddings from the selected model, and its tokenizer
def get_embeddings(texts, tokenizer, model, batch_size, max_length):
    model.eval()
    all_embeddings = []
    total = len(texts)

    # Sorting sentences by length for inference efficiency
    lengths = [len(t.split()) for t in texts]
    sorted_indices = np.argsort(lengths) # returns an array of sorted indices (not values)
    sorted_texts = [texts[i] for i in sorted_indices]

    for i in range(0, total, batch_size):
        batch = sorted_texts[i: i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            output = model(**encoded) # Outputs the logits, that need activations, use output.keys()

        cls_embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

        if (i // batch_size) % 10 == 0:
            percentage = (i / total) * 100
            print(f"{percentage:.2f}% processed...")

    all_embeddings = np.vstack(all_embeddings)
    restore_indices = np.argsort(sorted_indices)
    return all_embeddings[restore_indices]

In [5]:
# Functions to find the optimal batch_size for each model
sample_texts = X_train_txt[:512]

def test_batch_size(batch_size, tokenizer, model, max_length):
    try:
        batch = sample_texts[:batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            output = model(**encoded)

        process = psutil.Process(os.getpid())
        ram_used = process.memory_info().rss / 1e9

        if DEVICE == "mps":
            mps_used = torch.mps.current_allocated_memory() / 1e9
            mps_reserved= torch.mps.driver_allocated_memory() / 1e9
            print(f"    batch_size={batch_size: 4d}")
            print(f"RAM: {ram_used: .2f}GB")
            print(f"MPS allocated: {mps_used: .2f}GB")
            print(f"MPS reserved: {mps_reserved: .2f}GB | OK")
        else:
            print(f"    batch_size={batch_size:4d} | RAM: {ram_used: .2f}GB | OK")

        return True
    
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"    batch_size={batch_size:.4d} | OOM")
            if DEVICE == "mps":
                torch.mps.empty_cache()
            return False
        raise e
    
def find_optimal_batch_size(tokenizer, model, max_length):
    candidates = [64, 128, 256, 512]
    optimal = 64

    if model == "bert-large-uncased":
        candidates = [64, 128, 256]

    print(" Searching for optimal batch size...")
    print("="*60)

    for bs in candidates:
        success = test_batch_size(bs, tokenizer, model, max_length=max_length)
        if success:
            optimal = bs
        else:
            break

    print("=" * 60)
    print(f"    Max stable : {optimal}")
    return optimal

In [ ]:
# Trying different models for evaluation, using feature extraction approach.
for model_name in MODELS_TO_EVAL:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print('='*60)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)

    # Using the top 95th percentile as maximum to drop outliers, and prevent inference' errors
    lengths = [len(tokenizer.encode(t, truncation=False)) for t in df_sample["comment_text"]]
    p95 = int(np.percentile(lengths, 95))
    max_length = p95

    batch_size = find_optimal_batch_size(tokenizer, model, max_length)

    print("     Extracting training embeddings...")
    X_train_emb = get_embeddings(X_train_txt, tokenizer, model, batch_size, max_length)

    print("     Extracting testing embeddings...")
    X_test_emb = get_embeddings(X_test_txt, tokenizer, model, batch_size, max_length)

    del model, tokenizer
    torch.mps.empty_cache() # Drops reserved memory space

    clf = OneVsRestClassifier(LogisticRegression(max_iter=1500, random_state=42))
    clf.fit(X_train_emb, y_train)
    y_pred = clf.predict(X_test_emb)

    print(classification_report(y_test, y_pred, target_names=LABEL_COLS, zero_division=0))

    del X_train_emb, X_test_emb, clf
    torch.mps.empty_cache()

In [8]:
for model in MODELS_TO_EVAL:
    print(model)
    config = AutoConfig.from_pretrained(model)
    print(config.vocab_size)

roberta-base


50265
vinai/bertweet-base
64001
GroNLP/hateBERT
30522
distilbert-base-uncased
30522
bert-large-uncased
30522
